# `Credit Scoring Model`

### **Problem**
A bank wants to know, before giving a loan, whether a person is likely to pay it back.
In this project I predict whether a borrower will be **90+ days late on a payment within the next 2 years** (`SeriousDlqin2yrs = 1`) using their past financial history.

**Which mistake is worse?**
- If the model says a risky person is safe (False Negative), the bank gives the loan and may lose the money.
- If the model says a safe person is risky (False Positive), the bank only loses a customer.

So catching defaulters (**recall**) matters more than overall accuracy.

**Dataset:** [Give Me Some Credit](https://www.kaggle.com/datasets/brycecf/give-me-some-credit-dataset?select=cs-training.csv), Kaggle (`cs-training.csv`), 150,000 borrowers.

**Models:** Logistic Regression, Decision Tree, Random Forest
**Metrics:** Precision, Recall, F1-Score, ROC-AUC

# `Libraries`

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

TARGET = 'SeriousDlqin2yrs'

# `Data Loading & Understanding`

In [2]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
file_path = os.path.join(parent_dir, "data", "cs-training.csv")
df = pd.read_csv(file_path)
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

The dataset has 150,000 rows and 12 columns. All columns are numbers, so no mapping is needed.
- `Unnamed: 0` is only a row number, so I will drop it.<br>
- `MonthlyIncome` and `NumberOfDependents` have fewer non-null values than the other columns, so they have missing data.

In [4]:
df = df.drop(columns=['Unnamed: 0'])
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeriousDlqin2yrs,150000.0,0.066840,0.249746,0.0,0.000000,0.000000,0.000000,1.0
RevolvingUtilizationOfUnsecuredLines,150000.0,6.048438,249.755371,0.0,0.029867,0.154181,0.559046,50708.0
age,150000.0,52.295207,14.771866,0.0,41.000000,52.000000,63.000000,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.421033,4.192781,0.0,0.000000,0.000000,0.000000,98.0
DebtRatio,150000.0,353.005076,2037.818523,0.0,0.175074,0.366508,0.868254,329664.0
MonthlyIncome,120269.0,6670.221237,14384.674215,0.0,3400.000000,5400.000000,8249.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,150000.0,8.452760,5.145951,0.0,5.000000,8.000000,11.000000,58.0
NumberOfTimes90DaysLate,150000.0,0.265973,4.169304,0.0,0.000000,0.000000,0.000000,98.0
NumberRealEstateLoansOrLines,150000.0,1.018240,1.129771,0.0,0.000000,1.000000,2.000000,54.0
NumberOfTime60-89DaysPastDueNotWorse,150000.0,0.240387,4.155179,0.0,0.000000,0.000000,0.000000,98.0


**What I noticed in `describe()`:**
- `SeriousDlqin2yrs` mean is 0.068, so only **6.8% of people defaulted**. The data is imbalanced.
- `age` minimum is **0**, which is impossible for a borrower.
- The three late-payment columns have a max of **98**, but the 75th percentile is 0.
- `RevolvingUtilizationOfUnsecuredLines` should be between 0 and 1, but the max is **50,708**.
- `DebtRatio` median is 0.37 but the mean is 353, and `MonthlyIncome` goes up to 3 million while the median is 5,400. Both have some extremely large values.

# `Data Checking`
### Missing values

In [5]:
df.isnull().sum()

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

`MonthlyIncome` has 29,731 missing values (almost 20%) and `NumberOfDependents` has 3,924 (almost 2.6%). 
20% is too much to delete, so it will be filled later.

### Duplicates

In [6]:
duplicate = df[df.duplicated()]
duplicate.count()

SeriousDlqin2yrs                        609
RevolvingUtilizationOfUnsecuredLines    609
age                                     609
NumberOfTime30-59DaysPastDueNotWorse    609
DebtRatio                               609
MonthlyIncome                            99
NumberOfOpenCreditLinesAndLoans         609
NumberOfTimes90DaysLate                 609
NumberRealEstateLoansOrLines            609
NumberOfTime60-89DaysPastDueNotWorse    609
NumberOfDependents                      513
dtype: int64

### Age

In [7]:
df['age'].unique()

array([ 45,  40,  38,  30,  49,  74,  57,  39,  27,  51,  46,  76,  64,
        78,  53,  43,  25,  32,  58,  50,  69,  24,  28,  62,  42,  75,
        26,  52,  41,  81,  31,  68,  70,  73,  29,  55,  35,  72,  60,
        67,  36,  56,  37,  66,  83,  34,  44,  48,  61,  80,  47,  59,
        77,  63,  54,  33,  79,  65,  86,  92,  23,  87,  71,  22,  90,
        97,  84,  82,  91,  89,  85,  88,  21,  93,  96,  99,  94,  95,
       101,  98, 103, 102, 107, 105,   0, 109])

Most ages are between 21 and 109, but there is one **0**. A borrower cannot be 0 years old, so this is a **data entry error**.

### Late payment columns

In [9]:
LATE_COLS = []
for col in df.columns:
    if df[col].max() == 98:
        LATE_COLS.append(col)
LATE_COLS

['NumberOfTime30-59DaysPastDueNotWorse',
 'NumberOfTimes90DaysLate',
 'NumberOfTime60-89DaysPastDueNotWorse']